#Primary market data

In [0]:
%pip install yfinance

In [0]:
import yfinance as yf
import pandas as pd
from pyspark.sql import functions as F

In [0]:
tickers = {
    "^NSEI": "NIFTY_50",
    "^BSESN": "SENSEX",
    "^NSEBANK": "NIFTY_BANK",
    "^CNXENERGY": "NIFTY_ENERGY",
    "^CNXAUTO": "NIFTY_AUTO",
    "^CNXIT": "NIFTY_IT",
    "HAL.NS": "HAL",
    "INDIGO.NS": "INDIGO",
    "ASIANPAINT.NS": "ASIAN_PAINTS",
    "ONGC.NS": "ONGC",
    "BZ=F": "BRENT_CRUDE",
    "INR=X": "USD_INR",
    "GC=F": "GOLD",
    "^INDIAVIX": "INDIA_VIX"
}

In [0]:
dfs = []

for ticker, name in tickers.items():
    
    print(f"Downloading {ticker}")
    
    df = yf.download(
        ticker,
        start="2024-01-01",
        end="2025-04-01",
        interval="1d",
        progress=False
    )
    
    # ✅ CRITICAL FIX — flatten here
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    
    df = df.reset_index()
    df["ticker"] = ticker
    df["asset_name"] = name
    
    dfs.append(df)

In [0]:
final_df = pd.concat(dfs, ignore_index=True)

In [0]:
print(final_df.columns)


In [0]:
market_data= spark.createDataFrame(final_df)

In [0]:
market_data = market_data.select(
    F.col("Date").alias("trade_date"),
    F.col("Open").alias("open"),
    F.col("High").alias("high"),
    F.col("Low").alias("low"),
    F.col("Close").alias("close"),
    F.col("Volume").alias("volume"),
    "ticker",
    "asset_name"
)

In [0]:
display(market_data)

In [0]:
# Save market_data to landing zone as Parquet
market_data.write.mode("overwrite").parquet("/Volumes/iran_israel_capstone_project/bronze/landing_zone/market_data/")

